# Scoring threshold tuning

Calibrate `config/scoring.yaml` against batch metrics (and optional expert labels).

**Workflow**
1. Drop images under `data/raw/` (optional folders by posture type).
2. From the repo root, run the batch CLI:
   ```bash
   python -m technique_titan.batch.process_folder \
     --input data/raw --output data/processed --labels data/labels.csv
   ```
3. Run this notebook to inspect distributions, try candidate thresholds, and
   compare scores to labels. Copy approved values into `config/scoring.yaml`
   (do not overwrite that file from here until you are sure).

**Setup** (once, from repo root, Python 3.11 venv):
```bash
pip install -e .
pip install -r requirements-dev.txt
```

In [ ]:
from __future__ import annotations

import copy
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from technique_titan.scoring import load_config, score_all, score_metric

ROOT = Path.cwd()
if not (ROOT / "config" / "scoring.yaml").exists():
    # Allow running with cwd = notebooks/
    ROOT = ROOT.parent

CONFIG_PATH = ROOT / "config" / "scoring.yaml"
SUMMARY_PATH = ROOT / "data" / "processed" / "batch_summary.csv"

config = load_config(CONFIG_PATH)
criteria = config["criteria"]
bands = config["severity_bands"]

print(f"repo root : {ROOT}")
print(f"config    : {CONFIG_PATH}")
print(f"summary   : {SUMMARY_PATH} ({'found' if SUMMARY_PATH.exists() else 'missing — run batch CLI first'})")

## 1. Current thresholds

Each criterion scores **100** inside `ideal`, **0** at/beyond `limit`, and linearly in between.

In [ ]:
threshold_rows = []
for name, cfg in criteria.items():
    threshold_rows.append({
        "criterion": name,
        "metric": cfg["metric"],
        "ideal_lo": cfg["ideal"][0],
        "ideal_hi": cfg["ideal"][1],
        "limit_lo": cfg["limit"][0],
        "limit_hi": cfg["limit"][1],
        "weight": cfg["weight"],
    })
thresholds = pd.DataFrame(threshold_rows).set_index("criterion")
thresholds

## 2. Load batch summary

Expects `data/processed/batch_summary.csv` from the batch CLI (one row per detected hand).

In [ ]:
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"No batch summary at {SUMMARY_PATH}.\n"
        "Add images under data/raw/ then run:\n"
        "  python -m technique_titan.batch.process_folder "
        "--input data/raw --output data/processed"
    )

df = pd.read_csv(SUMMARY_PATH)
print(f"{len(df)} hand row(s), {df['source'].nunique()} image(s)")
df.head()

## 3. Metric distributions vs ideal / limit bands

Green = ideal (score 100). Orange dashed = limit edges (score 0).

In [ ]:
def plot_metric_bands(frame: pd.DataFrame, scoring_cfg: dict) -> None:
    items = list(scoring_cfg["criteria"].items())
    fig, axes = plt.subplots(len(items), 1, figsize=(9, 2.4 * len(items)), sharex=False)
    if len(items) == 1:
        axes = [axes]

    for ax, (name, cfg) in zip(axes, items):
        metric = cfg["metric"]
        if metric not in frame.columns:
            ax.set_title(f"{name}: missing column {metric}")
            continue
        values = frame[metric].dropna()
        ax.hist(values, bins=min(30, max(8, len(values) // 2)), color="#4a6fa5", alpha=0.85, edgecolor="white")
        ideal_lo, ideal_hi = cfg["ideal"]
        limit_lo, limit_hi = cfg["limit"]
        ax.axvspan(ideal_lo, ideal_hi, color="#4caf50", alpha=0.18, label="ideal")
        ax.axvline(limit_lo, color="#e67e22", ls="--", lw=1.5, label="limit")
        ax.axvline(limit_hi, color="#e67e22", ls="--", lw=1.5)
        ax.set_ylabel("count")
        ax.set_title(f"{name}  ({metric})  n={len(values)}")
        ax.legend(loc="upper right", fontsize=8)

    axes[-1].set_xlabel("metric value")
    fig.tight_layout()
    plt.show()

plot_metric_bands(df, config)
df[[c["metric"] for c in criteria.values()]].describe().T

## 4. What-if: candidate thresholds

Edit `CANDIDATE` below (only the fields you want to change). The next cells
rescore every row and compare composites / severity mixes to the current config.

In [ ]:
# Deep-copy current config, then override ranges you want to try.
CANDIDATE = copy.deepcopy(config)

# Examples (uncomment / edit):
# CANDIDATE["criteria"]["wrist_height"]["ideal"] = [-0.08, 0.18]
# CANDIDATE["criteria"]["wrist_height"]["limit"] = [-0.45, 0.55]
# CANDIDATE["criteria"]["finger_curvature"]["ideal"] = [110.0, 155.0]
# CANDIDATE["severity_bands"]["good_min"] = 75
# CANDIDATE["severity_bands"]["warning_min"] = 45

def rescore_frame(frame: pd.DataFrame, scoring_cfg: dict, prefix: str) -> pd.DataFrame:
    """Apply scoring.yaml logic to each row; returns score/severity/composite columns."""
    out = frame.copy()
    score_cols = {}
    sev_cols = {}
    composites = []

    for idx, row in frame.iterrows():
        metrics = {
            cfg["metric"]: row.get(cfg["metric"])
            for cfg in scoring_cfg["criteria"].values()
        }
        scored = score_all(metrics, scoring_cfg)
        for criterion, value in scored["scores"].items():
            score_cols.setdefault(criterion, []).append(value)
        for criterion, value in scored["severities"].items():
            sev_cols.setdefault(criterion, []).append(value)
        composites.append(scored["composite_score"])

    for criterion, values in score_cols.items():
        out[f"{prefix}_score_{criterion}"] = values
    for criterion, values in sev_cols.items():
        out[f"{prefix}_severity_{criterion}"] = values
    out[f"{prefix}_composite"] = composites
    return out

scored = rescore_frame(df, config, "current")
scored = rescore_frame(scored, CANDIDATE, "candidate")

compare = pd.DataFrame({
    "current_composite_mean": [scored["current_composite"].mean()],
    "candidate_composite_mean": [scored["candidate_composite"].mean()],
    "current_composite_median": [scored["current_composite"].median()],
    "candidate_composite_median": [scored["candidate_composite"].median()],
})
compare

In [ ]:
def severity_mix(frame: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = [c for c in frame.columns if c.startswith(f"{prefix}_severity_")]
    mixes = {}
    for col in cols:
        criterion = col.replace(f"{prefix}_severity_", "")
        mixes[criterion] = frame[col].value_counts(normalize=True).mul(100).round(1)
    return pd.DataFrame(mixes).fillna(0.0)

print("Current severity mix (% of hands)")
display(severity_mix(scored, "current"))
print("Candidate severity mix (% of hands)")
display(severity_mix(scored, "candidate"))

In [ ]:
# Largest composite swings under the candidate config
scored["composite_delta"] = scored["candidate_composite"] - scored["current_composite"]
cols = ["source", "hand", "current_composite", "candidate_composite", "composite_delta"]
scored.reindex(scored["composite_delta"].abs().sort_values(ascending=False).index)[cols].head(15)

## 5. Agreement with expert labels (optional)

If you passed `--labels data/labels.csv` to the batch CLI, summary rows include
`label_<criterion>` columns (`good` / `warning` / `critical`). This cell reports
match rate for current vs candidate severities.

In [ ]:
label_cols = [c for c in scored.columns if c.startswith("label_") and c != "label_notes" and c != "label_hand"]
criterion_from_label = {
    c: c.replace("label_", "")
    for c in label_cols
    if c.replace("label_", "") in criteria
}

if not criterion_from_label:
    print("No label_* severity columns found. Skip until you merge labels.csv in the batch run.")
else:
    rows = []
    for label_col, criterion in criterion_from_label.items():
        labeled = scored.dropna(subset=[label_col])
        if labeled.empty:
            continue
        current_col = f"current_severity_{criterion}"
        candidate_col = f"candidate_severity_{criterion}"
        rows.append({
            "criterion": criterion,
            "n_labeled": len(labeled),
            "current_match_%": (labeled[current_col] == labeled[label_col]).mean() * 100,
            "candidate_match_%": (labeled[candidate_col] == labeled[label_col]).mean() * 100,
        })
    agreement = pd.DataFrame(rows).round(1)
    display(agreement)
    print("Delta = candidate − current (positive means candidate matches labels more often)")
    display(agreement.assign(delta=agreement["candidate_match_%"] - agreement["current_match_%"]))

## 6. Score curve for one metric

Pick a criterion to see how raw metric values map to 0–100 under current vs candidate.

In [ ]:
CRITERION = "wrist_height"  # change as needed

cur_cfg = config["criteria"][CRITERION]
cand_cfg = CANDIDATE["criteria"][CRITERION]
metric = cur_cfg["metric"]
values = df[metric].dropna()
xs = pd.Series(
    sorted(
        set(values.tolist())
        | set(cur_cfg["ideal"] + cur_cfg["limit"] + cand_cfg["ideal"] + cand_cfg["limit"])
    )
)
# Dense grid across observed + threshold span
lo = min(xs.min(), cur_cfg["limit"][0], cand_cfg["limit"][0])
hi = max(xs.max(), cur_cfg["limit"][1], cand_cfg["limit"][1])
grid = pd.Series([lo + (hi - lo) * i / 200 for i in range(201)])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(grid, [score_metric(v, cur_cfg["ideal"], cur_cfg["limit"]) for v in grid], label="current", color="#2c3e50")
ax.plot(grid, [score_metric(v, cand_cfg["ideal"], cand_cfg["limit"]) for v in grid], label="candidate", color="#c0392b", ls="--")
ax.scatter(values, [score_metric(v, cur_cfg["ideal"], cur_cfg["limit"]) for v in values], s=18, alpha=0.45, color="#4a6fa5", label="batch (current score)")
ax.axhline(bands["good_min"], color="#4caf50", lw=1, alpha=0.7)
ax.axhline(bands["warning_min"], color="#e67e22", lw=1, alpha=0.7)
ax.set_xlabel(metric)
ax.set_ylabel("score")
ax.set_title(f"Score mapping — {CRITERION}")
ax.set_ylim(-5, 105)
ax.legend()
fig.tight_layout()
plt.show()

## 7. Export candidate YAML snippet

Prints the candidate `criteria` + `severity_bands` for manual paste into
`config/scoring.yaml` after you are happy with the what-if results.

In [ ]:
export = {
    "severity_bands": CANDIDATE["severity_bands"],
    "criteria": CANDIDATE["criteria"],
}
print(yaml.safe_dump(export, sort_keys=False))